In [ ]:
# W8 Day 3 - Blueprint → Compiler → ExecutionPlanIR
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
zh_font = font_manager.FontProperties(fname=font_path)
plt.rcParams['font.family'] = zh_font.get_name()
plt.rcParams['axes.unicode_minus'] = False
print(f'中文字体已配置: {zh_font.get_name()}')

# 🎯 今日学习目标｜第8周-Day3：Blueprint → Compiler → ExecutionPlanIR

> **链路第三步：意图怎么变成可执行的？**
>
> Agent Host 带着 ApplicationContract 来敲门了。LangChat 怎么把这个"做什么"的契约，变成一份机器可以确定性执行的运行计划？
> 这就是今天的主角：**制品链的核心三环——Blueprint → Compiler → ExecutionPlanIR**。

## 📅 学习进度

```text
W1  ████████████████████ ✅ Transformer 与大模型基础
W2  ████████████████████ ✅ Transformer 工程优化
W3  ████████████████████ ✅ 训练、SFT、RLHF、DPO
W4  ████████████████████ ✅ RAG 与知识增强
W5  ████████████████████ ✅ 推理与思维链
W6  ████████████████████ ✅ Agent 与工具使用
W7  ████████████████████ ✅ 数字员工架构深化
W8  ██████░░░░░░░░░░░░░░ 🔥 LangChat 心智模型（Day3/7）
W9  ░░░░░░░░░░░░░░░░░░░░ 🧩 领域对象深挖
W10 ░░░░░░░░░░░░░░░░░░░░ 🛡 Governance 横切约束
W11 ░░░░░░░░░░░░░░░░░░░░ 💻 代码现实与实施路线图
W12 ░░░░░░░░░░░░░░░░░░░░ 👁 Vision Intelligence 全景
W13 ░░░░░░░░░░░░░░░░░░░░ 🚀 视觉智能能力蓝图
```

**进度：第8周 / 第13周｜链路第三天——意图怎么变成可执行的。**

# 🔄 往期回顾

## W8 链路前两天

| Day | 主题 | 核心要点 | 与今天的关系 |
|-----|------|----------|--------------|
| Day1 | 用户意图 | Agent Host 直接调用 LangChat，不是 Orchestrator 编排 | Agent Host 的请求需要被翻译成可执行计划 |
| Day2 | ApplicationContract | 传输无关的业务契约，定义"做什么" | Contract 是 Blueprint 的输入约束，不是 Blueprint 本身 |

## W7 关联

| Day | 关联 |
|-----|------|
| W7-D3 任务编排 | Workflow 是"怎么做"，但 LangChat v2 制品链用 Blueprint + IR 取代了 WorkflowSpec |
| W7-D5 评估体系 | 评估发生在 Release Gate（ReleaseEvaluation），不发生在 Build 阶段 |

## 昨天的延续

昨天学到 ApplicationContract 是"做什么"的业务契约。今天进入下一个核心问题：

**有了契约之后，"怎么做"的执行逻辑从哪里来？谁来把设计意图翻译成机器可执行的表示？**

# 📚 第一部分：为什么 Blueprint（蓝图制品）不能直接运行？

## 🎯 今日核心问题

> **为什么 Blueprint 不能直接运行？**

### 生活类比：建筑设计图 vs 施工计划

想象你在盖一栋楼：

| 角色 | 建筑行业 | LangChat |
|------|----------|----------|
| 设计草图 | 建筑师的手绘概念图 | BlueprintCandidate |
| 审核通过的设计图 | 盖章的施工蓝图 | BlueprintVersion |
| 施工组织计划 | 分阶段施工方案（先地基、再框架、再装修） | ExecutionPlanIR |
| 盖好的楼 | 交付的可入住建筑 | SkillRelease v2 |

**你能拿设计图直接盖楼吗？不能。**设计图说"这里要有一面墙"，但施工计划还需要决定：先浇混凝土还是先砌砖？哪个工序可以并行？哪些材料要提前采购？

同理，Blueprint 是人可读的、声明式的、描述"要做什么"的制品；ExecutionPlanIR 是机器可执行的、过程式的、描述"怎么做"的内部表示。中间的转换过程就是 Compiler（编译器）。

### 三个关键对象的边界

| 对象 | 层级 | 可人工编辑？ | 可直接执行？ | 可直接部署？ |
|------|------|-------------|-------------|-------------|
| BlueprintCandidate | 作者提交的草案 | ✅ 是（Draft 阶段） | ❌ 否 | ❌ 否 |
| BlueprintVersion | 评审通过的快照 | ❌ 否（不可变） | ❌ 否 | ❌ 否 |
| ExecutionPlanIR | Compiler 产物 | ❌ 否（不可变） | ❌ 否（需 SkillRelease 包装） | ❌ 否（需 DeploymentRevision） |

**链上每一环不可跳过、不可逆序**（ADR-005 §9，HC-3）。

### 为什么不能跳过 Compiler？

如果 Blueprint 可以直接运行，意味着：
1. **设计即执行**——每次运行的逻辑取决于谁写了 Blueprint，没有标准化编译过程
2. **没有确定性**——同一份 Blueprint 在不同环境可能跑出不同结果
3. **没有可审计性**——无法在 Build 阶段做 Policy Check、Resolve、Optimize
4. **没有可复现性**——缺少 Compiler 版本+参数+依赖锁的完整输入身份

Compiler 的存在不是多余的翻译层，而是**确定性的保证**：同一完整输入身份永远产出同一 ExecutionPlanIR digest。

In [ ]:
# 制品链全景可视化：从 Candidate 到 Runtime
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('LangChat v2 制品链：从意图到执行', fontsize=18, fontweight='bold', pad=20)

# 定义节点位置
nodes = [
    (1.5, 7, 'BlueprintCandidate\n（作者草案）', '#FFE0B2', '#E65100', '可编辑\n不可执行\n不可部署'),
    (4.5, 7, 'BlueprintVersion\n（评审通过）', '#C8E6C9', '#1B5E20', '不可变\n不可执行\n不可部署'),
    (4.5, 4, 'Build\n（输入身份）', '#BBDEFB', '#0D47A1', '完整输入声明\n确定性参数+依赖锁'),
    (8, 4, 'Compiler\n10 阶段流水线', '#E1BEE7', '#4A148C', 'Parse → Validate →\nNormalize → Resolve →\nPolicy → Plan → Lower →\nOptimize → Package'),
    (11.5, 4, 'ExecutionPlanIR\n（执行计划）', '#FFCCBC', '#BF360C', '不可变\n不可直接执行\n不可直接部署'),
    (11.5, 7, 'SkillRelease v2\n（唯一可部署制品）', '#B2DFDB', '#004D40', 'OCI 制品\nIR + 依赖锁 + Source Map'),
    (14.5, 7, 'DeploymentRevision\n（运行时闭包）', '#C5CAE9', '#1A237E', 'digest-pin 全部依赖\n可执行'),
]

for x, y, label, fc, ec, desc in nodes:
    box = mpatches.FancyBboxPatch((x-1.2, y-0.8), 2.4, 1.6, 
                                   boxstyle='round,pad=0.2',
                                   facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x, y+0.3, label, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(x, y-0.4, desc, ha='center', va='center', fontsize=7, color='#555')

# 箭头连接
arrows = [
    (2.7, 7, 3.3, 7, '评审通过\n(Admission +\nSource Review)'),
    (4.5, 6.2, 4.5, 4.8, '输入'),
    (5.7, 4, 6.8, 4, '编译'),
    (9.2, 4, 10.3, 4, '产出'),
    (11.5, 4.8, 11.5, 6.2, '打包'),
    (12.7, 7, 13.3, 7, '物化'),
]
for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', lw=2.5, color='#333'))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx+0.15, my+0.15, label, fontsize=7, color='#1565C0', fontstyle='italic')

# 今天重点区域高亮
highlight = mpatches.FancyBboxPatch((3.3, 2.8), 9, 3.4, 
                                     boxstyle='round,pad=0.3',
                                     facecolor='none', edgecolor='#D32F2F', 
                                     linewidth=2.5, linestyle='--')
ax.add_patch(highlight)
ax.text(7.8, 2.5, '⬆ 今天重点：Blueprint → Compiler → ExecutionPlanIR', 
       ha='center', fontsize=11, color='#D32F2F', fontweight='bold')

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第8周/w8d3_artifact_chain.png', dpi=150, bbox_inches='tight')
plt.show()
print('制品链全景图已保存')

# 📚 第二部分：ADR-005 如何设计？

## D-2：BlueprintCandidate 评审流程

ADR-005 §5 定义了 Candidate → BlueprintVersion 的两段式入口：

### 第一段：Admission Validation（机器判定）

| 检查项 | 判定规则 | 失败行为 |
|--------|----------|----------|
| 结构完整性 | canonical bytes 能被解析为合法 schema | 拒绝，不进入评审 |
| 引用合法性 | ApplicationContractVersion digest 存在且有引用权 | 拒绝 |
| Policy floor 合规 | effect_policy 不放宽最低约束 | 拒绝 |
| Self-Reference 禁止 | bytes 不包含自身 digest | 拒绝 |
| 内容寻址可计算 | 能产出稳定 digest | 拒绝 |

**关键：Admission 是机器判定，不允许人工覆盖。**

### 第二段：Source Review（人工 + 规则组合）

| Reviewer 类 | 职责 | 强制性 |
|-------------|------|--------|
| 结构评审 | 复核 Admission 未覆盖的结构风险 | 强制 |
| 合规评审 | 复核 Policy、scope、人审门配置 | 强制 |
| 业务评审 | 提供业务建议 | 可选，不阻塞 |

**关键：评审只看结构/引用/合规，不评业务正确性。** 业务正确性归 ReleaseEvaluation（Domain Model §7.12 SC-19）。

## D-3：Build / BuildRun Compiler 版本治理

### Compiler 的 10 个阶段（Artifact Spec §7.2）

```text
BlueprintVersion bytes
    ↓
 1. Parse        — 解析为 AST
 2. Validate     — 结构校验 + Compatibility Matrix 三点检查
 3. Normalize    — 规范化 AST
 4. Resolve      — 解析依赖为精确 digest
 5. Policy Check — Policy floor 合规
 6. Plan         — 确定性调度计划（不调 LLM！）
 7. Lower        — 降低为 IR 中间形式
 8. Optimize     — 确定性优化
 9. Package      — 组装 ExecutionPlanIR + SourceMap
10. Provenance   — 组装构建证据
    ↓
ExecutionPlanIR + SourceMap + ProvenanceManifest
```

### Compiler 版本号：EPOCH.MAJOR.MINOR

| 递增类型 | 含义 | 示例 |
|----------|------|------|
| EPOCH | 阶段大重构（阶段重排、IR schema 不兼容） | 1.0.0 → 2.0.0 |
| MAJOR | 单阶段不兼容变更 | 1.0.0 → 1.1.0 |
| MINOR | bug 修复，行为不变 | 1.0.0 → 1.0.1 |

**同一版本号的 Compiler 实现字节级确定。**

### Build 的铁律（ADR-005 §6.1, HC-7）

- Build **MUST NOT** 进行 Release 评估
- Build **MUST NOT** 调用 LLM 进行 Planning / Authoring / IR 重写
- Build **MUST NOT** 以 WorkflowSpec 为输入（HC-2）
- Build **MUST NOT** 进行 Production signing

一句话：**Build 只负责确定性编译，不做价值判断。**

## D-4：ExecutionPlanIR 的可读性边界

| 场景 | 是否允许 |
|------|----------|
| 通过 Source Map 回溯 IR 节点到源位置 | ✅ 允许 |
| 通过 Registry 按 digest 拉 IR bytes 用于离线分析 | ✅ 允许 |
| 在 Evaluation 中作为评估对象子结构 | ✅ 允许 |
| 人工编辑 IR bytes | ❌ 禁止 |
| 直接执行 IR（不通过 SkillRelease 包装） | ❌ 禁止 |
| 直接部署 IR（不通过 DeploymentRevision） | ❌ 禁止 |
| 把 IR 作为 wire 对外暴露 | ❌ 禁止 |

In [ ]:
# Compiler 10 阶段流水线可视化
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('Compiler 10 阶段确定性 Build 流水线（Artifact Spec §7.2）', 
             fontsize=16, fontweight='bold', pad=15)

stages = [
    ('1.\nParse', '#E3F2FD', '解析为 AST'),
    ('2.\nValidate', '#BBDEFB', '结构校验\n+ Matrix 检查'),
    ('3.\nNormalize', '#90CAF9', '规范化 AST'),
    ('4.\nResolve', '#64B5F6', '解析依赖\n→ 精确 digest'),
    ('5.\nPolicy', '#42A5F5', 'Policy floor\n合规检查'),
    ('6.\nPlan', '#2196F3', '调度计划\n（不调 LLM）'),
    ('7.\nLower', '#1E88E5', '降低为\nIR 中间形式'),
    ('8.\nOptimize', '#1565C0', '确定性优化'),
    ('9.\nPackage', '#0D47A1', '组装 IR\n+ SourceMap'),
    ('10.\nProvenance', '#01579B', '构建证据\n（detached）'),
]

stage_w = 1.3
stage_gap = 0.25
start_x = 0.5

for i, (name, color, desc) in enumerate(stages):
    x = start_x + i * (stage_w + stage_gap)
    box = mpatches.FancyBboxPatch((x, 2), stage_w, 2, boxstyle='round,pad=0.1',
                                   facecolor=color, edgecolor='#333', linewidth=1)
    ax.add_patch(box)
    text_color = '#FFF' if i >= 5 else '#333'
    ax.text(x + stage_w/2, 3.3, name, ha='center', va='center', 
           fontsize=10, fontweight='bold', color=text_color)
    ax.text(x + stage_w/2, 2.5, desc, ha='center', va='center', 
           fontsize=7, color=text_color)
    if i < 9:
        ax.annotate('', xy=(x + stage_w + stage_gap, 3), xytext=(x + stage_w, 3),
                   arrowprops=dict(arrowstyle='->', lw=1.5, color='#666'))

# 输入和输出
ax.text(0.5 + 5*(stage_w+stage_gap)/2 - stage_gap/2, 5, 'BlueprintVersion (immutable digest)', 
       ha='center', fontsize=10, fontweight='bold', color='#1B5E20',
       bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', edgecolor='#1B5E20'))

ax.annotate('', xy=(0.5 + 0*(stage_w+stage_gap) + stage_w/2, 4.2), 
           xytext=(0.5 + 5*(stage_w+stage_gap)/2 - stage_gap/2, 4.7),
           arrowprops=dict(arrowstyle='->', lw=2, color='#1B5E20'))

# 输出
output_x = start_x + 10 * (stage_w + stage_gap) - stage_gap
ax.text(output_x/2 + 0.5, 0.8, '输出：ExecutionPlanIR + SourceMap + ProvenanceManifest', 
       ha='center', fontsize=10, fontweight='bold', color='#BF360C',
       bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFCCBC', edgecolor='#BF360C'))

# 禁区标注
forbidden = mpatches.FancyBboxPatch((1, -0.2), 14, 0.7, 
                                     boxstyle='round,pad=0.1',
                                     facecolor='#FFEBEE', edgecolor='#C62828', 
                                     linewidth=1.5, linestyle='--')
ax.add_patch(forbidden)
ax.text(8, 0.15, '❌ Build 禁区：不评估 Release · 不审批 · 不签发 · 不调 LLM · 不以 WorkflowSpec 为输入', 
       ha='center', fontsize=8, color='#C62828', fontweight='bold')

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第8周/w8d3_compiler_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Compiler 流水线图已保存')

# 📚 第三部分：现有代码如何实现？

## 3.1 BlueprintCandidate（草案阶段）

代码位置：`apps/backend/langchat/blueprint/candidate.py`

```python
@dataclass(frozen=True)
class BlueprintCandidate:
    candidate_id: str
    tenant_id: str
    workspace_id: str
    blueprint_id: str
    application_contract_version_digest: str  # ← 绑定 Contract
    content_digest: str                       # ← 内容寻址
    effect_policy: str                         # ← 策略声明
    lineage_parent_ids: tuple[str, ...]        # ← 谱系追溯
    state: str = "draft"                       # ← 生命周期
```

**生命周期**：`Draft → In Review → Promoted | Rejected`（终态不可逆）

### Admission 入口

代码位置：`apps/backend/langchat/blueprint/admission.py`

```python
def admit(candidate: BlueprintCandidate) -> AdmissionDecision:
    # (1) 结构完整性 — 检查必填字段
    # (2) 引用合法性 — ContractVersion digest 格式校验
    # (3) Policy floor — effect_policy 在允许集合内
    # ⚠️ 不评估业务正确性！(ADR-005 D-2)
```

## 3.2 BlueprintVersion（不可变源制品）

代码位置：`apps/backend/langchat/blueprint/version.py`

```python
@dataclass(frozen=True)
class BlueprintVersion:
    blueprint_id: str
    version: str
    tenant_id: str
    workspace_id: str
    application_contract_version_digest: str
    originating_candidate_id: str   # ← 谱系指针（必须非空！）
    content_digest: str
    state: str = "active"           # ← 创建即 Active，没有 Draft
```

**生命周期**：`Active → Deprecated → Retired`（无 Draft 态）

**关键规则**：BlueprintVersion 创建后不可修改（Domain Model §7.3 SC-03）。任何修改请求 = 新 Candidate = 重新走完整入口。

## 3.3 Build / BuildRun（确定性编译）

代码位置：`apps/backend/langchat/supply_chain/build.py`

```python
@dataclass(frozen=True)
class Build:
    build_id: str
    blueprint_version_digest: str     # ← 必须 BlueprintVersion，不能是 WorkflowSpec
    compiler_version: str             # ← EPOCH.MAJOR.MINOR
    deterministic_params: Mapping[str, object]
    dependency_lock: Mapping[str, str]  # ← 精确 digest，不允许 range
    target_compat_key: CompatCellKey   # ← 兼容性矩阵维度
```

**`input_digest` 排除了 `build_id`**——两个不同 build_id 但相同输入身份的 Build 产出相同 digest。这就是确定性。

### WorkflowSpec 拦截器

```python
def validate_build_input(value: object) -> None:
    """拒绝 WorkflowSpec 类型的输入"""
    for module_name in modules_to_check:
        if module_name.startswith("langchat.workflow."):
            raise InvalidBuildInputError(
                code="workflowspec-not-supported",
                message="Build input must be a BlueprintVersion..."
            )
```

这段代码在 Build 入口**显式拒绝** WorkflowSpec——是 ADR-005 HC-2 的代码级实现。

## 3.4 10 阶段流水线（pipeline.py + stages.py）

代码位置：`apps/backend/langchat/supply_chain/pipeline.py` + `stages.py`

`run_pipeline()` 函数按固定顺序执行 10 个阶段，每个阶段都是纯函数：

```python
STAGE_ORDER = (
    "parse", "validate", "normalize", "resolve", "policy_check",
    "plan", "lower", "optimize", "package", "provenance"
)
```

每个阶段接收 `StageContext`，返回 `StageResult`（含 accumulated output + Provenance entry）。

## 3.5 ExecutionPlanIR（不可编辑内部表示）

代码位置：`apps/backend/langchat/supply_chain/execution_plan_ir.py`

```python
IR_VENDOR_MEDIA_TYPE = "application/vnd.langchat.execution-plan-ir.v1+json"
IR_SCHEMA_VERSION = "v1"

@dataclass(frozen=True)
class IRNode:
    node_id: str
    node_type: str
    payload_digest: str       # ← 语义内容的 SHA-256
    source_position: str      # ← 回溯到 Blueprint 源位置

@dataclass(frozen=True)
class ExecutionPlanIR:
    ir_schema_version: str    # ← 固定 "v1"
    nodes: tuple[IRNode, ...] # ← 不可变 tuple
```

**digest 只取决于 `(ir_schema_version, nodes)`**——时间戳、build_run_id、Operator 身份全部排除在 hashed bytes 之外。

In [ ]:
# 演示：用 Python 模拟 Blueprint → Build → ExecutionPlanIR 完整链路

import hashlib
import json
from dataclasses import dataclass, field
from typing import Literal

# ========================================
# 1. BlueprintCandidate（草案）
# ========================================
@dataclass(frozen=True)
class BlueprintCandidate:
    candidate_id: str
    tenant_id: str
    workspace_id: str
    blueprint_id: str
    contract_version_digest: str
    content: str  # 简化：用字符串代表内容
    effect_policy: str = "read_only"
    state: str = "draft"
    
    @property
    def content_digest(self):
        return 'sha256:' + hashlib.sha256(self.content.encode()).hexdigest()

# ========================================
# 2. BlueprintVersion（不可变制品）
# ========================================
@dataclass(frozen=True)
class BlueprintVersion:
    blueprint_id: str
    version: str
    originating_candidate_id: str
    contract_version_digest: str
    content_digest: str
    state: str = "active"

# ========================================
# 3. Build（输入身份）
# ========================================
@dataclass(frozen=True)
class Build:
    build_id: str
    blueprint_version_digest: str
    compiler_version: str  # EPOCH.MAJOR.MINOR
    deterministic_params: dict
    dependency_lock: dict
    
    @property
    def input_digest(self):
        # 注意：build_id 不参与 input_digest 计算！
        payload = {
            'blueprint_version_digest': self.blueprint_version_digest,
            'compiler_version': self.compiler_version,
            'deterministic_params': self.deterministic_params,
            'dependency_lock': self.dependency_lock,
        }
        canonical = json.dumps(payload, sort_keys=True, separators=(',',':'))
        return 'sha256:' + hashlib.sha256(canonical.encode()).hexdigest()

# ========================================
# 4. ExecutionPlanIR（编译产物）
# ========================================
@dataclass(frozen=True)
class IRNode:
    node_id: str
    node_type: str
    payload_digest: str
    source_position: str

@dataclass(frozen=True)
class ExecutionPlanIR:
    ir_schema_version: str  # 固定 "v1"
    nodes: tuple
    
    @property
    def digest(self):
        payload = {
            'ir_schema_version': self.ir_schema_version,
            'nodes': [{'node_id': n.node_id, 'node_type': n.node_type,
                       'payload_digest': n.payload_digest,
                       'source_position': n.source_position} for n in self.nodes]
        }
        canonical = json.dumps(payload, sort_keys=True, separators=(',',':'))
        return 'sha256:' + hashlib.sha256(canonical.encode()).hexdigest()

# ========================================
# 5. 简化版 Compiler（10 阶段 → 3 步演示）
# ========================================
def simulate_build(build):
    """简化版：模拟 Parse → Validate → Package"""
    print(f"  [阶段1 Parse]    解析 BlueprintVersion content...")
    print(f"  [阶段2 Validate] 校验结构 + Compatibility Matrix...")
    print(f"  [阶段9 Package]  组装 ExecutionPlanIR...")
    
    node = IRNode(
        node_id='root',
        node_type='entry',
        payload_digest=build.input_digest,  # 简化
        source_position='blueprint.md:L1'
    )
    ir = ExecutionPlanIR(ir_schema_version='v1', nodes=(node,))
    return ir

# ========================================
# 演示完整链路
# ========================================
print("=" * 65)
print("Blueprint → Compiler → ExecutionPlanIR 完整链路演示")
print("=" * 65)

# Step 1: 作者提交 Candidate
print("\n📝 Step 1: 作者提交 BlueprintCandidate")
candidate = BlueprintCandidate(
    candidate_id='cand-001',
    tenant_id='tenant-a',
    workspace_id='ws-1',
    blueprint_id='bp-mall-service',
    contract_version_digest='sha256:abc123contract',
    content='''skill: mall-internal-service
  input:
    message: string
  steps:
    - search_knowledge_base
    - generate_response''',
)
print(f"  candidate_id: {candidate.candidate_id}")
print(f"  content_digest: {candidate.content_digest[:20]}...")
print(f"  state: {candidate.state}")

# Step 2: Admission 校验
print("\n🔍 Step 2: Admission Validation（机器判定）")
print(f"  ✅ 结构完整性: 通过")
print(f"  ✅ 引用合法性: contract_version_digest 格式正确")
print(f"  ✅ Policy floor: effect_policy='read_only' 在允许集合内")
print(f"  → Candidate 进入 In Review")

# Step 3: Source Review → 升级为 BlueprintVersion
print("\n📋 Step 3: Source Review 通过 → 物化为 BlueprintVersion")
bp_version = BlueprintVersion(
    blueprint_id='bp-mall-service',
    version='v1',
    originating_candidate_id=candidate.candidate_id,
    contract_version_digest=candidate.contract_version_digest,
    content_digest=candidate.content_digest,
)
bp_digest = 'sha256:' + hashlib.sha256(
    json.dumps({'bp_id': bp_version.blueprint_id, 'v': bp_version.version,
               'content': bp_version.content_digest}).encode()
).hexdigest()
print(f"  blueprint_id: {bp_version.blueprint_id}")
print(f"  version: {bp_version.version}")
print(f"  digest: {bp_digest[:20]}...")
print(f"  state: {bp_version.state}（创建即 Active，无 Draft）")

# Step 4: Build 输入身份
print("\n🔨 Step 4: 创建 Build（完整输入身份）")
build = Build(
    build_id='build-001',
    blueprint_version_digest=bp_digest,
    compiler_version='1.0.0',
    deterministic_params={'optimization_level': 'standard'},
    dependency_lock={'knowledge_snapshot': 'sha256:ks-001',
                    'policy_bundle': 'sha256:pb-001'},
)
print(f"  build_id: {build.build_id}")
print(f"  compiler_version: {build.compiler_version}")
print(f"  input_digest: {build.input_digest[:20]}...")
print(f"  ⚠️ build_id 不参与 input_digest 计算！")

# Step 5: Compiler 编译
print("\n⚙️ Step 5: Compiler 执行 10 阶段")
ir = simulate_build(build)
print(f"  ir_schema_version: {ir.ir_schema_version}")
print(f"  nodes: {len(ir.nodes)}")
print(f"  ir.digest: {ir.digest[:20]}...")

# Step 6: 确定性验证
print("\n🔬 Step 6: 确定性验证——同输入，同输出")
build2 = Build(
    build_id='build-002',  # ← 不同 build_id！
    blueprint_version_digest=bp_digest,
    compiler_version='1.0.0',
    deterministic_params={'optimization_level': 'standard'},
    dependency_lock={'knowledge_snapshot': 'sha256:ks-001',
                    'policy_bundle': 'sha256:pb-001'},
)
ir2 = simulate_build(build2)
print(f"  build-001.input_digest: {build.input_digest[:30]}...")
print(f"  build-002.input_digest: {build2.input_digest[:30]}...")
print(f"  相同？ {build.input_digest == build2.input_digest}")
print(f"  build-001 IR digest: {ir.digest[:30]}...")
print(f"  build-002 IR digest: {ir2.digest[:30]}...")
print(f"  相同？ {ir.digest == ir2.digest}")
print(f"  ✅ 确定性验证通过！build_id 不同但输入身份相同 → IR digest 相同")

In [ ]:
# 差距分析：当前态与目标态可视化
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('差距分析：制品链当前态与目标态', 
             fontsize=16, fontweight='bold', pad=20)

# 左侧：当前态
ax.text(3, 9.3, '当前态（WP-03 阶段）', ha='center', fontsize=13, fontweight='bold', color='#E65100')
current_box = mpatches.FancyBboxPatch((0.5, 1), 5, 7.8, boxstyle='round,pad=0.3',
                                       facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(current_box)
current_items = [
    ('✅', 'BlueprintCandidate 数据类'),
    ('✅', 'BlueprintVersion 数据类'),
    ('✅', 'Admission Validation 函数'),
    ('✅', 'Build / BuildRun 数据类'),
    ('✅', 'ExecutionPlanIR + IRNode 数据类'),
    ('✅', '10 阶段流水线框架（stages.py）'),
    ('✅', 'WorkflowSpec 拦截器（HC-2 代码级）'),
    ('✅', '确定性 digest 计算逻辑'),
    ('🟡', '10 阶段大多为 pass-through stub'),
    ('🔴', 'BlueprintCandidate 不支持外部导入'),
    ('🔴', 'Source Review 流程未实现'),
    ('🔴', 'Compiler 各阶段为占位，无真实编译'),
]
for i, (icon, text) in enumerate(current_items):
    color = '#4CAF50' if icon == '✅' else ('#FF9800' if icon == '🟡' else '#F44336')
    ax.text(1, 8.2 - i * 0.6, f'{icon} {text}', fontsize=9, color=color)

# 右侧：目标态
ax.text(10.75, 9.3, '目标态（ADR-005 + AS）', ha='center', fontsize=13, fontweight='bold', color='#1B5E20')
target_box = mpatches.FancyBboxPatch((8.25, 1), 5, 7.8, boxstyle='round,pad=0.3',
                                      facecolor='#E8F5E9', edgecolor='#1B5E20', linewidth=2)
ax.add_patch(target_box)
target_items = [
    ('📋', 'External Authoring Client 导入器'),
    ('📋', '完整 Admission Validation 规则集'),
    ('📋', 'Source Review 多角色评审流程'),
    ('📋', 'Compiler 版本注册（EPOCH.MAJOR.MINOR）'),
    ('📋', '10 阶段真实编译实现'),
    ('📋', 'ExecutionPlanIR 字段级 schema'),
    ('📋', 'Source Map 完整映射'),
    ('📋', 'Provenance SLSA-style 证据链'),
    ('📋', 'SkillRelease v2 OCI 打包'),
    ('📋', 'Release Gate（Evaluation + Approval + Signature）'),
    ('📋', 'DeploymentRevision 闭包物化'),
    ('📋', 'WorkflowSpec cutover & retire'),
]
for i, (icon, text) in enumerate(target_items):
    ax.text(8.75, 8.2 - i * 0.6, f'{icon} {text}', fontsize=9, color='#333')

# 中间箭头
ax.annotate('', xy=(8.25, 5), xytext=(5.5, 5),
            arrowprops=dict(arrowstyle='->', lw=3, color='#1565C0'))
ax.text(6.875, 5.5, 'WP-03 → WP-04+\n渐进式演进', ha='center', fontsize=10, 
       color='#1565C0', fontweight='bold')

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第8周/w8d3_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('差距分析图表已保存')

# 📚 第四部分：业务关联

## 制品链在 LangChat 完整链路中的位置

```
用户意图
    ↓
Agent Host（理解意图，决定调用哪个能力）
    ↓
ApplicationContract（W8-D2：定义"做什么"）
    ↓
BlueprintCandidate（作者提交草案）
    ↓ Admission Validation + Source Review
BlueprintVersion（不可变源制品）
    ↓ ← 今天在这里！Compiler 10 阶段
BuildRun（确定性编译）
    ↓
ExecutionPlanIR（内部不可编辑执行计划）
    ↓ + Dependency Lock + Source Map + Behavioral Assets
SkillRelease v2（唯一可部署 OCI 制品）
    ↓ Release Gate: Evaluation + Approval + Signature + Publication
DeploymentRevision（运行时闭包）
    ↓
Runtime（在 FrozenExecutionContext 内执行）
```

## 为什么业务需要关心制品链？

| 业务场景 | 制品链的作用 |
|---------|-------------|
| 新增能力（如"客流异常预警"） | 作者在 EAC 提交 Candidate → 评审 → BlueprintVersion → Build → SkillRelease |
| 修改逻辑（如调整 prompt） | 修改 = 新 Candidate = 新 BlueprintVersion = 新 BuildRun = 新 SkillRelease |
| 回滚到旧版本 | 前向语义：物化新 DeploymentRevision 指向历史 SkillRelease digest |
| 审计追溯 | Source Map 把 IR 节点映射回 Blueprint 源位置，Provenance 记录完整构建链 |
| 多环境部署 | 同一 SkillRelease digest 跨环境部署，环境差异在 DeploymentRevision 闭包 |

## 📘 今天多理解了什么？

**以前以为：** Blueprint 就是一份配置文件，Runtime 直接读取执行。

**现在知道：**
1. Blueprint 是**制品**（artifact），不是配置——它有版本、digest、生命周期
2. Blueprint 和 ExecutionPlanIR 之间隔着 10 阶段确定性 Compiler——不是简单翻译
3. Compiler 的存在不是多余：它保证了**同一输入永远产出同一输出**
4. ExecutionPlanIR 是**内部不可编辑**的——不存在"手动 patch IR"的合法路径
5. Build 阶段**不做价值判断**（不评估 Release、不调 LLM、不签发）——职责严格分离

## 🔮 反问：如果今天重新设计，还会这样做吗？

会。而且理由比昨天更充分。

原因：
1. **确定性 Build 是信任的基础。** 如果 Blueprint 可以直接运行，每次执行结果取决于解释器的偶然行为，无法审计、无法复现。
2. **Compiler 版本治理让工具链可演进。** Compiler 升级不会偷偷改变已发布 Release 的行为。
3. **ExecutionPlanIR 不可编辑是不可谈判的。** 一旦允许"IR hotfix"，整个确定性链条就断了——谁改的？什么时候改的？为什么改的？不可追溯。
4. **Build 禁区（不调 LLM、不评估 Release）是关注点分离的典范。** 编译就是编译，评估就是评估。混在一起就两边都做不好。

# 📝 每日工程日志

| 类型 | 内容 |
|------|------|
| **新增** | 理解了制品链的完整链路：Candidate → Admission → Source Review → BlueprintVersion → Build → Compiler 10 阶段 → ExecutionPlanIR → SkillRelease |
| **新增** | 理解了 Compiler 的 10 个阶段（Parse → Validate → Normalize → Resolve → Policy Check → Plan → Lower → Optimize → Package → Provenance）及其确定性约束 |
| **新增** | 理解了 ExecutionPlanIR 的可读性边界：允许查看（Source Map 回溯、Registry 拉取、Evaluation 引用），禁止任何写入 |
| **修改** | 以前认为 Blueprint 是配置文件；现在知道 Blueprint 是制品（artifact），有 digest、版本、生命周期 |
| **确认** | `Build.input_digest` 排除 `build_id`——不同 build_id 但相同输入身份产出相同 IR digest（确定性） |
| **确认** | WorkflowSpec 拦截器（`validate_build_input`）是 ADR-005 HC-2 的代码级实现 |
| **遗留** | 当前 10 阶段流水线大多为 pass-through stub（WP-03 阶段），无真实编译逻辑 |
| **遗留** | BlueprintCandidate 不支持从 External Authoring Client 导入（WorkflowSpec 导入器未实现） |
| **技术债** | Source Review 流程（结构评审 + 合规评审）未实现；当前只有 Admission Validation |
| **技术债** | Compiler 各阶段版本号硬编码为 1.0.0，无独立版本治理 |
| **下一步** | 明天学习 Runtime：执行计划怎么跑起来的？为什么 Runtime 不保存状态？ |

# 🔑 今日英文术语（10个）

| # | 英文术语 | 音标 | 中文释义 |
|---|----------|------|----------|
| 1 | **Blueprint** | /ˈbluːprɪnt/ | 设计制品——人可读的、声明式的、描述"要做什么"的 canonical 源制品 |
| 2 | **BlueprintCandidate** | /ˈbluːprɪnt ˈkændɪdət/ | 蓝图候选——作者提交的草案，经评审后升级为 BlueprintVersion |
| 3 | **BlueprintVersion** | /ˈbluːprɪnt ˈvɜːrʒn/ | 蓝图版本——不可变、内容寻址的 canonical 源制品 |
| 4 | **Compiler** | /kəmˈpaɪlər/ | 编译器——把 BlueprintVersion 确定性编译为 ExecutionPlanIR 的 10 阶段流水线 |
| 5 | **ExecutionPlanIR** | /ˌeksɪˈkjuːʃn plæn aɪˈɑːr/ | 执行计划中间表示——Compiler 产出的内部不可编辑表示 |
| 6 | **BuildRun** | /bɪld rʌn/ | 构建运行——Build 的一次具体执行，产出 IR + SourceMap + Provenance |
| 7 | **Deterministic Build** | /dɪˌtɜːrmɪˈnɪstɪk bɪld/ | 确定性构建——同一完整输入身份永远产出相同产出 digest |
| 8 | **Provenance** | /ˈprɒvənəns/ | 溯源证据——记录构建全链路的 detached attestation |
| 9 | **Source Map** | /sɔːrs mæp/ | 源映射——把 IR 节点映射回 BlueprintVersion 源位置 |
| 10 | **Immutable** | /ɪˈmjuːtəbl/ | 不可变——对象创建后内容不可修改，是制品链的核心约束 |

# ✏️ 课堂练习

## 练习1：判断对错

1. BlueprintVersion 可以被人工修改后重新保存。 → ___
2. 同一 BlueprintVersion + 同一 Compiler 版本，产出相同 ExecutionPlanIR digest。 → ___
3. ExecutionPlanIR 可以被 Runtime 直接装载执行。 → ___
4. Build 阶段可以调用 LLM 来优化 prompt。 → ___
5. WorkflowSpec 可以作为 Build 的输入。 → ___

<details>
<summary>📌 点击展开答案</summary>

1. **错** — BlueprintVersion 创建后不可修改（Domain Model §7.3 SC-03）。修改 = 新 Candidate。
2. **对** — 确定性构建：同一完整输入身份 → 同一产出 digest（ADR-005 D-3）。
3. **错** — ExecutionPlanIR 不可直接执行，必须通过 SkillRelease 包装 + DeploymentRevision 物化。
4. **错** — Build MUST NOT 调用 LLM（ADR-005 HC-7、AS §7.3、§18.3-5）。
5. **错** — WorkflowSpec MUST NOT 作为 Build 输入（HC-2）。
</details>

## 练习2：排列制品链顺序

请把以下对象按制品链正确顺序排列：

```
A. ExecutionPlanIR
B. BlueprintCandidate
C. DeploymentRevision
D. SkillRelease v2
E. BlueprintVersion
F. BuildRun
```

正确顺序：___ → ___ → ___ → ___ → ___ → ___

# 📝 课后测试

**Q1:** 为什么 BlueprintVersion 不能直接运行？
- A) 因为它没有输入输出定义
- B) 因为它是人可读的声明式制品，需要经过 Compiler 确定性编译为 ExecutionPlanIR 才能执行
- C) 因为它的版本号不对
- D) 因为它没有 digest

**Q2:** Compiler 的 10 个阶段中，"Plan"阶段可以使用 LLM 来生成执行计划吗？
- A) 可以，LLM 能生成更好的计划
- B) 不可以，Build MUST NOT 调用 LLM 进行 Planning（AS §7.3）
- C) 可以，但只在特殊情况下
- D) 只在 Release Gate 阶段可以

**Q3:** 以下哪个描述了 ExecutionPlanIR 的正确边界？
- A) 可执行、可编辑、可部署
- B) 不可执行、不可编辑、不可直接部署，但允许人工查看
- C) 可执行但不允许查看
- D) 可编辑但不可部署

**Q4:** Build 的 `input_digest` 包含以下哪个字段？
- A) build_id
- B) 构建时间戳
- C) compiler_version
- D) Operator 身份

**Q5:** 开放题：如果你是 Compiler 设计者，为什么你会把 `build_run_id` 排除在 hashed bytes 之外？

<details>
<summary>📌 点击展开答案</summary>

1. **B** — BlueprintVersion 是声明式制品，需 Compiler 编译为 IR 才能执行
2. **B** — Build 禁区明确禁止 LLM 调用（HC-7）
3. **B** — IR 可查看但不可编辑、不可直接执行、不可直接部署
4. **C** — compiler_version 是输入身份的一部分；build_id、时间戳、Operator 都被排除
5. 开放题参考：因为 `build_run_id` 是运行标识，不是输入身份。两次不同 build_run（不同 ID）如果输入完全相同，应该产出相同 IR digest。如果把 build_run_id 放入 hashed bytes，同一输入就会因 run ID 不同而产出不同 digest，破坏确定性。
</details>

# 📝 今日核心总结

## 一句话总结

> **Blueprint 是设计图，ExecutionPlanIR 是施工计划，Compiler 是把前者变成后者的确定性翻译机——同一设计图永远产出同一施工计划，这就是信任的基石。**

## 知识卡片

```
┌─────────────────────────────────────────────────────────┐
│              制品链核心三环                                │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  BlueprintVersion（设计制品）                             │
│    ├── 人可读、声明式                                     │
│    ├── 不可变（创建即 Active，无 Draft）                   │
│    └── 绑定 ApplicationContractVersion                   │
│                                                         │
│  Compiler（确定性编译器）                                  │
│    ├── 10 阶段流水线                                      │
│    ├── 版本号 EPOCH.MAJOR.MINOR                           │
│    ├── 同输入 → 同输出（确定性）                           │
│    └── 禁区：不调 LLM、不评估、不签发                      │
│                                                         │
│  ExecutionPlanIR（内部执行计划）                           │
│    ├── 不可变、内容寻址                                   │
│    ├── 不可直接执行（需 SkillRelease 包装）                │
│    ├── 不可直接部署（需 DeploymentRevision 闭包）          │
│    └── 允许查看，禁止任何写入                              │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

## 📎 真实参考

- ADR-005 §5（D-2）：BlueprintCandidate 评审流程
- ADR-005 §6（D-3）：Build / BuildRun Compiler 版本治理
- ADR-005 §7（D-4）：ExecutionPlanIR 内部性与可读性边界
- ADR-005 §9：单一制品链权威边界总图
- Artifact Spec §7.2：Build 阶段定义
- Domain Model §7.2 SC-02 / §7.3 SC-03 / §7.4 SC-04,05 / §7.5 SC-06
- `apps/backend/langchat/blueprint/candidate.py`
- `apps/backend/langchat/blueprint/version.py`
- `apps/backend/langchat/blueprint/admission.py`
- `apps/backend/langchat/supply_chain/build.py`
- `apps/backend/langchat/supply_chain/execution_plan_ir.py`
- `apps/backend/langchat/supply_chain/stages.py`
- `apps/backend/langchat/supply_chain/pipeline.py`